In [1]:
import os
import psycopg2

from astropy import units as u
from astropy.table import Table, join, unique, vstack
from astropy.coordinates import SkyCoord, match_coordinates_sky, search_around_sky

from desitarget.targets import decode_targetid, encode_targetid, resolve
from desitarget.io import releasedict, release_to_photsys

from tqdm.notebook import tqdm_notebook

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt


from astropy.table import Table, vstack, hstack

import psycopg2

from tqdm.notebook import tqdm_notebook

import time

### load 2020 sga 

In [2]:
sga = Table.read('/global/cfs/cdirs/cosmo/data/sga/2020/SGA-2020.fits', 'ELLIPSE')
sga[:5]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
2,SGA-2020 2,PGC1283207,1283207,228.3770865,5.4232017,S?,152.2,0.36307806,0.724436,0.03463229,23.40448,16.976,False,LEDA-20181114,0,PGC1283207,1,True,228.3770865,5.4232017,0.36307806,2283p055,228.3770803831908,5.423191398593787,0.49470574,SB26,158.20142,0.545691,228.37700918822188,5.4232652570544015,10.897086,3.3509698,3.1147978,3.240862,5.902337,6.9126143,7.941369,8.997992,10.073601,11.199986,12.391357,13.561038,14.841172,16.966799,16.108246,15.486356,16.879545,16.024958,15.400715,16.818878,15.967034,15.341793,16.776297,15.925804,15.300776,16.746685,15.897334,15.272053,16.725166,15.876816,15.2521105,16.708357,15.862035,15.237181,16.696539,15.851936,15.226998,16.689613,15.844313,15.21976,0.013392451,0.02354,0.021872982,0.01736985,0.024445537,0.039866067,0.05026544,0.08455789,0.122911856,0.005682776,0.0054258136,0.0049038026,0.005588406,0.005323561,0.0047632363,0.00543534,0.005177031,0.0046343105,0.0053025587,0.005040888,0.0045181247,0.005206092,0.0049438984,0.0044374703,0.0051483097,0.0048758644,0.0043834248,0.0051032505,0.0048264163,0.004344248,0.0050705094,0.004792021,0.004319857,0.005054293,0.004765629,0.0043044444,16.65942,0.34037337,0.2978292,3.0239506,0.07928849,15.820566,0.2640441,0.34559453,3.3033552,0.003811298,15.195567,0.29826432,0.3001073,3.2333765,0.011723555,0
3,SGA-2020 3,PGC1310416,1310416,202.54443750000002,6.9345944,Sc,159.26,0.4017908,0.7816278,0.073888786,23.498482,16.85,False,LEDA-20181114,1,PGC1310416,1,True,202.54443750000002,6.9345944,0.4017908,

### query function

In [32]:
def get_tf_targets_modified(redux, use_cached=False, verbose=False):
    """Get TF targets from the DESI observations DB for a given spectroscopic reduction.
    
    Parameters
    ----------
    redux : str
        Spectroscopic reduction. E.g., 'everest', 'fuji', ...
    use_cached : bool
        Use cached data rather than re-running the query.
    
    Returns
    -------
    
    tf_targets : Table
        Table of Tully-Fisher observations.
    """
    tf_targets = None

    if os.path.exists(f'tf_targets_{redux}.fits') and use_cached:
        tf_targets = Table.read('cache/tf_targets_{redux}.fits')
    else:
        try:
            db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
            # db = psycopg2.connect(host='decatdb.lbl.gov', database='desidb', user='desi', password='')
            cursor = db.cursor()

            # query = f"""SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, zd.z, zd.zerr, zd.spectype, zd.deltachi2, zd.zwarn, pv.pvtype, pv.sga_id
            #        FROM {redux}.healpix_fibermap as rdx, static.pv as pv, {redux}.healpix_redshifts as zd
            #        WHERE q3c_join(rdx.target_ra, rdx.target_dec, pv.ra, pv.dec, 1./3600.) 
            #              AND zd.targetid = rdx.targetid;"""
                         # AND pv.sga_id IS NOT NULL AND (pv.pvtype LIKE 'TFT' or pv.pvtype LIKE 'EXT' or pv.pvtype LIKE 'SGA');"""


            # should select targets within sga d26 (converted to radius in deg) + 1 arcsec for buffer
            # query = f"""SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, zd.z, zd.zerr, zd.spectype,
            # zd.deltachi2, zd.zwarn, sga.sga_id, sga.ra, sga.dec, sga.d26
            #        FROM {redux}.healpix_fibermap as rdx, static.sga as sga, {redux}.healpix_redshifts as zd
            #        WHERE q3c_radial_query(rdx.target_ra, rdx.target_dec, 234.40098578623957, 5.973890372677745, (5.0586667/120. + 1./3600.)) 
            #              AND (zd.targetid = rdx.targetid);"""

            query = f"""SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, sga.sga_id, sga.ra, sga.dec, sga.d26
                   FROM {redux}.healpix_fibermap as rdx, static.sga as sga, {redux}.healpix_redshifts as zd
                   WHERE q3c_radial_query(221.076522, 0.04404,rdx.target_ra, rdx.target_dec, 1.) 
                        ;"""


            
            
            if verbose:
                print(query)

            cursor.execute(query)

            if verbose:
                print('executed')
            
            rows = cursor.fetchall()
            print(rows)

            if verbose:
                print('rows fetched')
            
            tf_targets = Table(list(map(list, zip(*rows))),
                               names=['TARGETID', 'TARGET_RA', 'TARGET_DEC', 'Z', 'ZERR', 'SPECTYPE', 
                                      'DELTACHI2', 'ZWARN', 'SGA_ID', 'SGA_RA', 'SGA_DEC', 'SGA_D26'])

            print('table created')

            # #- Select only targets with SGA IDs and PV types matching SGA, EXT, and TFT
            # select = (tf_targets['SGA_ID'] != None) & \
            #          ((tf_targets['PVTYPE'] == 'TFT') | \
            #           (tf_targets['PVTYPE'] == 'EXT') | \
            #           (tf_targets['PVTYPE'] == 'SGA'))
            # tf_targets = tf_targets[select]

            #- Use TARGETID to extract the photometric system used during targeting
            # _, _, releases, _, _, _ = decode_targetid(tf_targets['TARGETID'].value)

            # photsys = []
            # for i, release in enumerate(releases):
            #     ps = None

            #     if release in releasedict:
            #         ps = release_to_photsys([release])[0].decode('utf-8')
            #     else:
            #         #- Fall-through case: not all SGA center observations are in the main survey.
            #         #  In this case, select 'N' or 'S' based on the SGA object's position.
            #         ra  = tf_targets['TARGET_RA'][i]
            #         dec = tf_targets['TARGET_DEC'][i]
            #         c = SkyCoord(ra=ra, dec=dec, unit='degree')

            #         #- N: in galactic northern hemisphere and with dec > 32.375. Else, S.
            #         isnorth = (c.galactic.b > 0) & (dec > 32.375)
            #         ps = 'N' if isnorth else 'S'

            #     photsys.append(ps)

            # #- Complain if the photsys table doesn't match the size of the Vrot table.
            # if len(photsys) != len(tf_targets):
            #     print(f'photsys array of len {len(photsys)} != targets array of len {len(tf_targets)}')

            # tf_targets['PHOTSYS'] = photsys

            # tf_targets.write(f'cache/tf_targets_{redux}.fits', overwrite=True)

        except Exception as error:
            print(error)
        finally:
            if db is not None:
                db.close()

    return tf_targets

In [33]:
t = get_tf_targets_modified('loa', verbose=1)

SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, sga.sga_id, sga.ra, sga.dec, sga.d26
                   FROM loa.healpix_fibermap as rdx, static.sga as sga, loa.healpix_redshifts as zd
                   WHERE q3c_radial_query(221.076522, 0.04404,rdx.target_ra, rdx.target_dec, 1./120.) 
                        ;
executed
[]
rows fetched
table created


In [34]:
t

TARGETID,TARGET_RA,TARGET_DEC,Z,ZERR,SPECTYPE,DELTACHI2,ZWARN,SGA_ID,SGA_RA,SGA_DEC,SGA_D26
float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64


In [9]:
sga[sga['SGA_ID']==1980]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
1980,SGA-2020 1980,PGC3098317,3098317,221.078673,0.0419199,S0,141.12,0.9418896,0.699842,0.028935684,23.985483,15.487,False,LEDA-20181114,560,PGC3098317_GROUP,2,True,221.0777837382504,0.04279262980888227,1.0166478,2211p000,221.0765229904383,0.04404351172445308,1.0488012,SB26,164.451,0.9800691,221.07686987533287,0.04358679757899937,32.102997,9.238959,9.879681,10.428718,7.8350058,10.399587,13.1465645,15.987329,18.755962,21.488228,24.465588,27.493338,31.464037,15.995904,15.606668,15.309033,15.664592,15.287069,15.01805,15.44377,15.06274,14.800893,15.315058,14.9204,14.647455,15.237978,14.831263,14.5480585,15.184224,14.767017,14.474368,15.144503,14.717999,14.417657,15.118924,14.688098,14.377791,15.099152,14.662473,14.347829,0.06076624,0.08439203,0.06663399,0.058722753,0.07866875,0.059216432,0.09420997,0.12256561,0.22303835,0.020017084,0.020355469,0.021976111,0.015951209,0.016249087,0.01809202,0.013424219,0.013525385,0.015170244,0.011982458,0.011943107,0.013248542,0.011183206,0.0110309515,0.012122214,0.010671716,0.01040515,0.011336535,0.010279736,0.009965059,0.010787699,0.010061269,0.009693397,0.010398413,0.009885436,0.00947659,0.010130541,15.039355,0.6984608,1.5859108,2.528108,0.259893,14.587156,0.56408924,2.7104928,2.587596,0.27126014,14.262344,0.40249574,6.1888504,2.868194,0.17947721,0


In [31]:
db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT * 
            FROM static.sga
            WHERE 1=0;"""

cursor.execute(query)

rows = cursor.fetchall()
print(rows)

db.close()


[]


## find a target in loa

In [35]:
loa_tf = Table.read('/global/cfs/cdirs/desi/science/td/pv/tfgalaxies/Y3/desi_pv_loa_healpix.fits')
loa_tf[:5]

TARGETID,TARGET_RA,TARGET_DEC,Z,ZERR,SPECTYPE,DELTACHI2,ZWARN,PVTYPE,SGA_ID,PHOTSYS
int64,float64,float64,float64,float64,bytes6,float64,int64,bytes3,int64,bytes1
-430502046,134.00013295228783,5.934552839997555,1.5211812148930146,8.936834292063904e-05,GALAXY,22.838147282600403,2049,TFT,838970,S
-427872363,61.981718642816155,-22.823913110732356,0.049086072011651605,8.769259809810332e-06,GALAXY,320.07210237160325,2049,TFT,982213,S
-411444222,156.2181500220905,7.1584828480845655,0.7655255485455215,0.0001486141274558249,GALAXY,3.784611649578437,2053,TFT,4614,S
-261707523,51.18809619857989,-15.380001206793725,0.11707091119461288,1.2658057107331122e-05,GALAXY,5004.164042882621,512,SGA,788458,S
-260779407,138.32747528464944,17.5639019859795,0.08569268559472955,9.449907878073889e-06,GALAXY,237.52352766692638,2560,TFT,735997,S


In [9]:
sga[sga['SGA_ID'] == 838970]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32
838970,SGA-2020 838970,SDSSJ085600.31+055604.4,3452702,134.001357,5.9345324,E?,83.78,0.33419505,0.36307806,0.060180627,24.59048,18.342,False,LEDA-20181114,305978,SDSSJ085600.31+055604.4,1,True,134.001357,5.9345324,0.33419505,1339p060,134.00131760677266,5.934559958075137,0.4177438,LEDA,84.52326,0.32225138,134.0012375486832,5.934627764152796,6.492079,-1.0,-1.0,-1.0,5.025152,6.358781,7.912158,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,18.302841,17.409012,16.804663,18.1562,17.228645,16.649128,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.03206396,0.06857974,0.11101419,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0.021706618,0.024167256,0.028484741,0.02213439,0.021406587,0.025650296,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,0


In [13]:
db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, zd.z, sga.d26
                   FROM loa.healpix_fibermap as rdx, loa.healpix_redshifts as zd, static.sga as sga
                   WHERE q3c_radial_query(rdx.target_ra, rdx.target_dec, 134.00131760677266, 5.934559958075137, 0.4177438/120. + 1./3600.) 
                        AND rdx.targetid = zd.targetid AND sga.sga_id = 838970;"""

cursor.execute(query)

rows = cursor.fetchall()
print(rows)

db.close()


[]


In [17]:
db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT rdx.targetid, rdx.target_ra, rdx.target_dec, zd.z, sga.d26
                   FROM loa.healpix_fibermap as rdx, loa.healpix_redshifts as zd, static.sga as sga
                   WHERE q3c_join(sga.ra, sga.dec, rdx.target_ra, rdx.target_dec, sga.d26/120. + 1./3600.) 
                        AND rdx.targetid = zd.targetid AND sga.sga_id < 10.;"""

cursor.execute(query)

rows = cursor.fetchall()
print(rows)

db.close()

[]


In [64]:
db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT *
            FROM static.sga;"""

cursor.execute(query)

rows = cursor.fetchone()
# print(rows)

db.close()

In [65]:
rows

In [67]:
db = psycopg2.connect(host='desidb-rr.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT pv.ra
            FROM static.pv as pv
;
"""

cursor.execute(query)

rows = cursor.fetchall()
# print(rows)

db.close()

In [68]:
rows[:5]

[(226.62204270638287,),
 (226.62323955692426,),
 (226.62443636990747,),
 (226.40142777879575,),
 (226.40426477759235,)]

In [70]:
db = psycopg2.connect(host='decatdb.lbl.gov', database='desidb', user='desi', password='')
cursor = db.cursor()

query = f"""SELECT sga.sga_id
            FROM static.sga as sga
;
"""

cursor.execute(query)

rows = cursor.fetchall()
# print(rows)

db.close()

In [3]:
sga[sga['SGA_ID'] == 5053801]

SGA_ID,SGA_GALAXY,GALAXY,PGC,RA_LEDA,DEC_LEDA,MORPHTYPE,PA_LEDA,D25_LEDA,BA_LEDA,Z_LEDA,SB_D25_LEDA,MAG_LEDA,BYHAND,REF,GROUP_ID,GROUP_NAME,GROUP_MULT,GROUP_PRIMARY,GROUP_RA,GROUP_DEC,GROUP_DIAMETER,BRICKNAME,RA,DEC,D26,D26_REF,PA,BA,RA_MOMENT,DEC_MOMENT,SMA_MOMENT,G_SMA50,R_SMA50,Z_SMA50,SMA_SB22,SMA_SB22.5,SMA_SB23,SMA_SB23.5,SMA_SB24,SMA_SB24.5,SMA_SB25,SMA_SB25.5,SMA_SB26,G_MAG_SB22,R_MAG_SB22,Z_MAG_SB22,G_MAG_SB22.5,R_MAG_SB22.5,Z_MAG_SB22.5,G_MAG_SB23,R_MAG_SB23,Z_MAG_SB23,G_MAG_SB23.5,R_MAG_SB23.5,Z_MAG_SB23.5,G_MAG_SB24,R_MAG_SB24,Z_MAG_SB24,G_MAG_SB24.5,R_MAG_SB24.5,Z_MAG_SB24.5,G_MAG_SB25,R_MAG_SB25,Z_MAG_SB25,G_MAG_SB25.5,R_MAG_SB25.5,Z_MAG_SB25.5,G_MAG_SB26,R_MAG_SB26,Z_MAG_SB26,SMA_SB22_ERR,SMA_SB22.5_ERR,SMA_SB23_ERR,SMA_SB23.5_ERR,SMA_SB24_ERR,SMA_SB24.5_ERR,SMA_SB25_ERR,SMA_SB25.5_ERR,SMA_SB26_ERR,G_MAG_SB22_ERR,R_MAG_SB22_ERR,Z_MAG_SB22_ERR,G_MAG_SB22.5_ERR,R_MAG_SB22.5_ERR,Z_MAG_SB22.5_ERR,G_MAG_SB23_ERR,R_MAG_SB23_ERR,Z_MAG_SB23_ERR,G_MAG_SB23.5_ERR,R_MAG_SB23.5_ERR,Z_MAG_SB23.5_ERR,G_MAG_SB24_ERR,R_MAG_SB24_ERR,Z_MAG_SB24_ERR,G_MAG_SB24.5_ERR,R_MAG_SB24.5_ERR,Z_MAG_SB24.5_ERR,G_MAG_SB25_ERR,R_MAG_SB25_ERR,Z_MAG_SB25_ERR,G_MAG_SB25.5_ERR,R_MAG_SB25.5_ERR,Z_MAG_SB25.5_ERR,G_MAG_SB26_ERR,R_MAG_SB26_ERR,Z_MAG_SB26_ERR,G_COG_PARAMS_MTOT,G_COG_PARAMS_M0,G_COG_PARAMS_ALPHA1,G_COG_PARAMS_ALPHA2,G_COG_PARAMS_CHI2,R_COG_PARAMS_MTOT,R_COG_PARAMS_M0,R_COG_PARAMS_ALPHA1,R_COG_PARAMS_ALPHA2,R_COG_PARAMS_CHI2,Z_COG_PARAMS_MTOT,Z_COG_PARAMS_M0,Z_COG_PARAMS_ALPHA1,Z_COG_PARAMS_ALPHA2,Z_COG_PARAMS_CHI2,ELLIPSEBIT
int64,bytes16,bytes29,int64,float64,float64,bytes21,float32,float32,float32,float32,float32,float32,bool,bytes13,int64,bytes35,int16,bool,float64,float64,float32,bytes8,float64,float64,float32,bytes4,float32,float32,float64,float64,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,float32,int32


In [5]:
sga2025 = Table.read('/global/cfs/cdirs/cosmo/work/legacysurvey/sga/2025/SGA2025-beta-parent-refcat-v1.6.kd.fits')

In [6]:
sga2025[:5]

ra,dec,ref_id,mag,fitmode,pa,ba,diam
float64,float64,int64,float32,int32,float32,float32,float32
219.9258023922458,-32.53004546468686,4176048,17.78,0,70.35528,0.5832305,0.55550754
219.89199876009113,-32.34299358156585,4176016,17.71,0,150.42897,0.4607132,0.50789976
219.9409044773007,-32.331297660068095,242036,17.84,0,134.88263,0.22073081,0.68223655
220.20921498175156,-32.0109310161516,243654,18.04,0,91.79849,0.7038806,0.507167
220.3623232697446,-31.994910572523846,4257822,15.4,0,10.682078,0.6169077,1.0668384


In [9]:
sga2025[sga2025['ref_id'] ==5053803]

ra,dec,ref_id,mag,fitmode,pa,ba,diam
float64,float64,int64,float32,int32,float32,float32,float32
0.717,-60.83,5053803,15.36,6,27.0,0.61,22.32
